# AIIMS ECG-PPG Sync Debug Notebook (fixed)

This notebook keeps the **same file paths, sync variable names, and anchor index values** as your original version, but changes the PPG detection logic to avoid non-physiological PTT artifacts.

### Why this version
- The earlier approach used `argmax` in a broad `[0.1, 0.8] s` window after every R-peak.
- That often picks a late wave / motion artifact, producing unrealistically high PTT and floor/ceiling clipping.
- Here, we use:
  1. stable drift correction using relative-time anchor mapping,
  2. robust local peak detection (`find_peaks` + prominence),
  3. physiological gating of PTT.

### Physiological target window used
From common ECG-to-finger-PPG transit timing reported in literature and device studies, practical adult ranges are often around **0.15-0.35 s**; we use a conservative acceptance gate of **0.10-0.45 s**.


In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
import neurokit2 as nk


## 1) Keep the same paths and variable names


In [ ]:
ppg_csv = Path("1/Polar/E90F3223/Fateh Singh_108505253_E90F3223_13ppg.csv")
ecg_csv = Path("1/Holter/06-08-2025.csv")
meta_data_csv = Path("1/Polar/E90F3223/Fateh Singh_108505253_E90F3223_13metaData.csv")

ppg_csv_device_2 = Path("1/Polar/E9124620/Fateh Singh_108505253_E9124620_14ppg.csv")


In [ ]:
fs_ecg = 200

ppg_df = pd.read_csv(ppg_csv)
ecg_df = pd.read_csv(ecg_csv)
ppg_df_device_2 = pd.read_csv(ppg_csv_device_2)

ecg_signal = ecg_df.iloc[:, 2].to_numpy()
ppg_signal = ppg_df.iloc[:, 1].to_numpy()
ppg_signal_device_2 = ppg_df_device_2.iloc[:, 1].to_numpy()

# derive true PPG sampling rates from timestamps
fs_ppg = 1.0 / np.mean(np.diff(ppg_df["time"].to_numpy()) / 1e9)
fs_ppg_device_2 = 1.0 / np.mean(np.diff(ppg_df_device_2["time"].to_numpy()) / 1e9)

print(f"ECG fs={fs_ecg:.3f} Hz | PPG1 fs={fs_ppg:.3f} Hz | PPG2 fs={fs_ppg_device_2:.3f} Hz")


## 2) Keep the same sync anchor index values


In [ ]:
# Device 1 (E90F3223) anchors - SAME as original notebook
ppg_idx_1 = 14343639
ppg_idx_2 = 27155637

# ECG anchors - SAME as original notebook
ecg_idx_1 = 16331017
ecg_idx_2 = 30954051

# Device 2 (E9124620) anchors - SAME as original notebook
ppg_idx_1_device_2 = 14403716
ppg_idx_2_device_2 = 27272110


## 3) Signal cleaning and R-peak detection


In [ ]:
ecg_clean = nk.ecg_clean(ecg_signal, sampling_rate=fs_ecg, method="neurokit")
_, ecg_info = nk.ecg_peaks(ecg_clean, sampling_rate=fs_ecg)
rpeaks = ecg_info["ECG_R_Peaks"]

ppg_clean = nk.ppg_clean(ppg_signal, sampling_rate=fs_ppg, method="elgendi")
ppg_clean_device_2 = nk.ppg_clean(ppg_signal_device_2, sampling_rate=fs_ppg_device_2, method="elgendi")

print("R-peaks:", len(rpeaks))


## 4) Stable clock-map fit (relative-time domain)


In [ ]:
def fit_linear_clock_map_from_anchors(ppg_time_ns, ecg_time_ns, ppg_anchor_idx, ecg_anchor_idx):
    ppg_anchor_ns = ppg_time_ns[np.asarray(ppg_anchor_idx, dtype=int)]
    ecg_anchor_ns = ecg_time_ns[np.asarray(ecg_anchor_idx, dtype=int)]

    t_ref_ns = int(min(ppg_anchor_ns.min(), ecg_anchor_ns.min()))
    t_ppg_rel = (ppg_anchor_ns - t_ref_ns) / 1e9
    t_ecg_rel = (ecg_anchor_ns - t_ref_ns) / 1e9

    a, b_rel = np.polyfit(t_ppg_rel, t_ecg_rel, deg=1)
    return {
        "a": a,
        "b_rel": b_rel,
        "t_ref_ns": t_ref_ns,
        "ppg_anchor_ns": ppg_anchor_ns,
        "ecg_anchor_ns": ecg_anchor_ns,
    }


def resample_ppg_to_ecg_domain_stable(ppg_time_ns, ppg_clean_sig, ecg_time_ns, ppg_anchor_idx, ecg_anchor_idx):
    map_info = fit_linear_clock_map_from_anchors(ppg_time_ns, ecg_time_ns, ppg_anchor_idx, ecg_anchor_idx)

    t_ref_ns = map_info["t_ref_ns"]
    a = map_info["a"]
    b_rel = map_info["b_rel"]

    t_ppg_rel = (ppg_time_ns.astype(np.int64) - t_ref_ns) / 1e9
    t_ecg_rel = (ecg_time_ns.astype(np.int64) - t_ref_ns) / 1e9

    # invert map: t_ecg_rel = a * t_ppg_rel + b_rel
    t_ecg_as_ppg_rel = (t_ecg_rel - b_rel) / a
    ppg_resampled = np.interp(t_ecg_as_ppg_rel, t_ppg_rel, ppg_clean_sig)

    return ppg_resampled, map_info


## 5) Robust ECG-guided PPG peak detection (fix)

Key difference vs old code:
- old: choose `argmax` in a very broad window (`0.1-0.8 s`), which can lock onto noise/late waves.
- new: detect candidate peaks with prominence, then choose best candidate near expected delay.


In [ ]:
def robust_ecg_guided_ppg_peaks(
    rpeak_indices,
    ppg_resampled,
    fs_ecg,
    min_delay=0.10,
    max_delay=0.45,
    expected_delay=0.22,
    min_prom_factor=0.25,
):
    ppg_peak_indices = []
    matched_rpeaks = []

    min_s = int(min_delay * fs_ecg)
    max_s = int(max_delay * fs_ecg)

    for r_idx in rpeak_indices:
        start_idx = r_idx + min_s
        end_idx = r_idx + max_s
        if end_idx >= len(ppg_resampled) or start_idx < 0:
            continue

        window = ppg_resampled[start_idx:end_idx]
        if len(window) < 5:
            continue

        # robust local prominence threshold
        amp = np.nanpercentile(window, 95) - np.nanpercentile(window, 5)
        min_prom = max(1e-6, min_prom_factor * amp)

        cand, props = find_peaks(window, distance=int(0.08 * fs_ecg), prominence=min_prom)
        if len(cand) == 0:
            # fallback: conservative local max only when no peak candidates exist
            cand = np.array([np.argmax(window)])
            prom = np.array([0.0])
        else:
            prom = props.get("prominences", np.zeros(len(cand)))

        # score = proximity to expected delay + prominence
        delays = (cand + min_s) / fs_ecg
        score = np.abs(delays - expected_delay) - 0.15 * (prom / (np.max(prom) + 1e-8))
        best = cand[np.argmin(score)]

        ppg_idx = start_idx + int(best)
        ptt = (ppg_idx - r_idx) / fs_ecg
        if min_delay <= ptt <= max_delay:
            ppg_peak_indices.append(ppg_idx)
            matched_rpeaks.append(r_idx)

    return np.asarray(ppg_peak_indices, dtype=int), np.asarray(matched_rpeaks, dtype=int)


def mad_outlier_mask(x, thresh=5.0):
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med)) + 1e-9
    z = 0.6745 * (x - med) / mad
    return np.abs(z) <= thresh


## 6) Device 1: before/after drift correction and final PTT


In [ ]:
# BEFORE drift correction (naive interpolation in absolute timestamp domain)
t_ppg_abs = ppg_df["time"].to_numpy(np.int64) / 1e9
t_ecg_abs = ecg_df["time"].to_numpy(np.int64) / 1e9
ppg_abs_to_ecg = np.interp(t_ecg_abs, t_ppg_abs, ppg_clean)

ppg_pk_before, r_before = robust_ecg_guided_ppg_peaks(rpeaks, ppg_abs_to_ecg, fs_ecg)
ptt_before = (ppg_pk_before - r_before) / fs_ecg

# AFTER drift correction
ppg_resampled_device_1_stable, map_device_1 = resample_ppg_to_ecg_domain_stable(
    ppg_df["time"].to_numpy(np.int64),
    ppg_clean,
    ecg_df["time"].to_numpy(np.int64),
    [ppg_idx_1, ppg_idx_2],
    [ecg_idx_1, ecg_idx_2],
)

ppg_pk_after, r_after = robust_ecg_guided_ppg_peaks(rpeaks, ppg_resampled_device_1_stable, fs_ecg)
ptt_after = (ppg_pk_after - r_after) / fs_ecg

# remove residual outliers
mask_after = mad_outlier_mask(ptt_after, thresh=5.0)
ptt_after_clean = ptt_after[mask_after]
beat_time_after = r_after[mask_after] / fs_ecg

print("Device1 map a:", map_device_1["a"], " drift ppm:", (map_device_1["a"] - 1) * 1e6)
print("PTT before mean±std:", np.mean(ptt_before), np.std(ptt_before), "N=", len(ptt_before))
print("PTT after mean±std :", np.mean(ptt_after_clean), np.std(ptt_after_clean), "N=", len(ptt_after_clean))


In [ ]:
# trend check
coef_before = np.polyfit(r_before / fs_ecg, ptt_before, 1) if len(ptt_before) > 2 else [np.nan, np.nan]
coef_after = np.polyfit(beat_time_after, ptt_after_clean, 1) if len(ptt_after_clean) > 2 else [np.nan, np.nan]

fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax[0].plot(r_before / fs_ecg, ptt_before, '.', ms=1, alpha=0.35, color='tab:green')
if np.isfinite(coef_before[0]):
    x = r_before / fs_ecg
    ax[0].plot(x, coef_before[0] * x + coef_before[1], 'k', lw=2)
ax[0].set_title("Device 1: PTT vs time (before drift correction)")
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("PTT (s)")
ax[0].set_ylim(0.08, 0.8)
ax[0].grid(alpha=0.2)

ax[1].plot(beat_time_after, ptt_after_clean, '.', ms=1, alpha=0.35, color='tab:red')
if np.isfinite(coef_after[0]):
    x = beat_time_after
    ax[1].plot(x, coef_after[0] * x + coef_after[1], 'k', lw=2)
ax[1].set_title("Device 1: PTT vs time (after drift correction)")
ax[1].set_xlabel("Time (s)")
ax[1].set_ylim(0.08, 0.8)
ax[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Slope before: {coef_before[0]:.3e} sec/sec")
print(f"Slope after : {coef_after[0]:.3e} sec/sec")


## 7) Device 2 (same strategy, same anchor variables)


In [ ]:
ppg_resampled_device_2_stable, map_device_2 = resample_ppg_to_ecg_domain_stable(
    ppg_df_device_2["time"].to_numpy(np.int64),
    ppg_clean_device_2,
    ecg_df["time"].to_numpy(np.int64),
    [ppg_idx_1_device_2, ppg_idx_2_device_2],
    [ecg_idx_1, ecg_idx_2],
)

ppg_pk_d2, r_d2 = robust_ecg_guided_ppg_peaks(rpeaks, ppg_resampled_device_2_stable, fs_ecg)
ptt_d2 = (ppg_pk_d2 - r_d2) / fs_ecg
mask_d2 = mad_outlier_mask(ptt_d2, thresh=5.0)
ptt_d2_clean = ptt_d2[mask_d2]

print("Device2 map a:", map_device_2["a"], " drift ppm:", (map_device_2["a"] - 1) * 1e6)
print("Device2 PTT mean±std:", np.mean(ptt_d2_clean), np.std(ptt_d2_clean), "N=", len(ptt_d2_clean))


## 8) Quick diagnostics for feasible PTT range


In [ ]:
def ptt_summary(name, ptt):
    q = np.nanpercentile(ptt, [1, 5, 25, 50, 75, 95, 99])
    print(f"\n{name}")
    print("  min/max:", np.min(ptt), np.max(ptt))
    print("  p01,p05,p25,p50,p75,p95,p99:", q)

ptt_summary("Device1 (after)", ptt_after_clean)
ptt_summary("Device2 (after)", ptt_d2_clean)

plt.figure(figsize=(8,4))
plt.hist(ptt_after_clean, bins=120, alpha=0.6, label='Device1')
plt.hist(ptt_d2_clean, bins=120, alpha=0.6, label='Device2')
plt.axvline(0.10, color='k', ls='--', lw=1)
plt.axvline(0.45, color='k', ls='--', lw=1)
plt.xlabel("PTT (s)")
plt.ylabel("Count")
plt.title("PTT distribution after robust detection + drift correction")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


### Interpretation checklist

If there are still issues after this fix, inspect these in order:
1. `map_device_1['a']` and `map_device_2['a']` should be close to 1 (small ppm drift).
2. `Slope after` should be much closer to zero than `Slope before`.
3. PTT distribution should cluster mostly in ~0.15-0.35 s (with some spread).
4. If PTT still piles at bounds 0.10 or 0.45, tighten motion filtering and/or revise expected delay.
